In [1]:
%pip install pandas pillow

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
from collections import Counter
from PIL import Image, UnidentifiedImageError
import pandas as pd
import statistics

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [6]:
# Internship project folder
PROJECT_ROOT = Path(r"C:\Users\sarun\OneDrive\Desktop\cesppl-internship")

# Dataset folder
RAW_DATASET = PROJECT_ROOT / "data" / "cesppl_raw"

print("Project Root :", PROJECT_ROOT)
print("Dataset Path :", RAW_DATASET)
print("Exists :", RAW_DATASET.exists())

Project Root : C:\Users\sarun\OneDrive\Desktop\cesppl-internship
Dataset Path : C:\Users\sarun\OneDrive\Desktop\cesppl-internship\data\cesppl_raw
Exists : True


In [7]:
if not RAW_DATASET.exists():
    raise FileNotFoundError(f"Dataset not found:\n{RAW_DATASET}")

folders = sorted(
    folder.name
    for folder in RAW_DATASET.iterdir()
    if folder.is_dir()
)

print("Dataset Found ✅")
print()

print("Number of folders:", len(folders))
print()

for folder in folders:
    print(folder)

Dataset Found ✅

Number of folders: 10

BIN LIFTING
BIN WASHING
GATE MEETING
LFC
MANUAL BEACH CLEANING
MECHANICAL SWEEPING
MECHANIZED BEACH CLEANING
PRIMARY COLLECTION
ROAD SWEEPING
SECONDARY VEHICLES


In [8]:
EXPECTED_COUNTS = {
    "BIN LIFTING":131,
    "BIN WASHING":327,
    "GATE MEETING":367,
    "LFC":217,
    "MANUAL BEACH CLEANING":1384,
    "MECHANICAL SWEEPING":284,
    "MECHANIZED BEACH CLEANING":302,
    "PRIMARY COLLECTION":208,
    "ROAD SWEEPING":530,
    "SECONDARY VEHICLES":394
}

print("Expected Images:", sum(EXPECTED_COUNTS.values()))

Expected Images: 4144


In [9]:
actual = {
    folder.name
    for folder in RAW_DATASET.iterdir()
    if folder.is_dir()
}

expected = set(EXPECTED_COUNTS.keys())

print("Missing:", expected-actual)
print("Extra:", actual-expected)

Missing: set()
Extra: set()


In [10]:
print("Folder Structure")
print("="*50)

total = 0

for cls in EXPECTED_COUNTS:

    folder = RAW_DATASET / cls

    files = [
        f for f in folder.rglob("*")
        if f.is_file()
    ]

    print(f"{cls:30} {len(files)}")

    total += len(files)

print("="*50)
print("Total Files:", total)

Folder Structure
BIN LIFTING                    166
BIN WASHING                    327
GATE MEETING                   367
LFC                            217
MANUAL BEACH CLEANING          1384
MECHANICAL SWEEPING            284
MECHANIZED BEACH CLEANING      302
PRIMARY COLLECTION             208
ROAD SWEEPING                  530
SECONDARY VEHICLES             394
Total Files: 4179


In [11]:
bin_lifting_folder = RAW_DATASET / "BIN LIFTING"

bin_lifting_files = sorted(
    file
    for file in bin_lifting_folder.rglob("*")
    if file.is_file()
)

print("Total files:", len(bin_lifting_files))
print("=" * 80)

for file in bin_lifting_files:
    print(file.name)

Total files: 166
image744.png
image745.png
image746.png
image747.png
image748.png
image749.png
image750.png
image751.png
image752.png
image753.png
image754.png
image755.png
image756.png
image757.png
image758.png
image759.png
image760.png
image761.png
image762.png
image763.png
image764.png
image765.png
image766.png
image767.png
image768.png
image769.png
image770.png
image771.png
image772.png
image773.png
image774.png
image775.png
image776.png
image777.png
image778.png
image779.png
image780.png
image781.png
image782.png
image783.png
image784.png
image785.png
image786.png
image787.png
image788.png
image789.png
image790.png
image791.jpeg
image792.png
image793.png
image794.png
image795.png
image796.png
image797.png
image798.png
image799.png
image800.png
image801.png
image802.png
image803.png
Screenshot 2026-05-31 133130.png
Screenshot 2026-05-31 133145.png
Screenshot 2026-05-31 133259.png
Screenshot 2026-05-31 133312.png
Screenshot 2026-05-31 133530.png
Screenshot 2026-05-31 133543.png
Scre

In [12]:
extension_counts = Counter(
    file.suffix.lower() if file.suffix else "[no extension]"
    for file in bin_lifting_files
)

print("BIN LIFTING files by extension")
print("=" * 40)

for extension, count in sorted(extension_counts.items()):
    print(f"{extension:20} {count}")

BIN LIFTING files by extension
.jpeg                75
.png                 91


In [13]:
valid_images = []
invalid_files = []

for file_path in bin_lifting_files:
    try:
        with Image.open(file_path) as image:
            image.verify()

        valid_images.append(file_path)

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        SyntaxError
    ) as error:
        invalid_files.append({
            "filename": file_path.name,
            "extension": file_path.suffix.lower() or "[no extension]",
            "reason": str(error)
        })

print("Total files   :", len(bin_lifting_files))
print("Valid images  :", len(valid_images))
print("Invalid files :", len(invalid_files))

Total files   : 166
Valid images  : 166
Invalid files : 0


In [14]:
invalid_df = pd.DataFrame(invalid_files)

if invalid_df.empty:
    print("No invalid or non-image files were found.")
else:
    display(invalid_df)

No invalid or non-image files were found.


In [15]:
import hashlib
from collections import defaultdict

def calculate_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(8192)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


hash_groups = defaultdict(list)

for file_path in valid_images:
    file_hash = calculate_hash(file_path)
    hash_groups[file_hash].append(file_path)

duplicate_groups = [
    files
    for files in hash_groups.values()
    if len(files) > 1
]

extra_duplicate_files = sum(
    len(group) - 1
    for group in duplicate_groups
)

print("Duplicate groups      :", len(duplicate_groups))
print("Extra duplicate files :", extra_duplicate_files)

for group_number, group in enumerate(duplicate_groups, start=1):
    print(f"\nDuplicate group {group_number}")

    for file_path in group:
        print("-", file_path.name)

Duplicate groups      : 0
Extra duplicate files : 0


In [16]:
png_files = sorted(
    file_path.name
    for file_path in valid_images
    if file_path.suffix.lower() == ".png"
)

jpeg_files = sorted(
    file_path.name
    for file_path in valid_images
    if file_path.suffix.lower() in {".jpg", ".jpeg"}
)

print("PNG files :", len(png_files))
print("JPEG files:", len(jpeg_files))

print("\nFirst 10 PNG files:")
for name in png_files[:10]:
    print("-", name)

print("\nFirst 10 JPEG files:")
for name in jpeg_files[:10]:
    print("-", name)

PNG files : 91
JPEG files: 75

First 10 PNG files:
- Screenshot 2026-05-31 133130.png
- Screenshot 2026-05-31 133145.png
- Screenshot 2026-05-31 133259.png
- Screenshot 2026-05-31 133312.png
- Screenshot 2026-05-31 133530.png
- Screenshot 2026-05-31 133543.png
- Screenshot 2026-05-31 133717.png
- Screenshot 2026-05-31 133745.png
- Screenshot 2026-05-31 134401.png
- Screenshot 2026-05-31 134454.png

First 10 JPEG files:
- WhatsApp Image 2026-04-22 at 1.29.06 PM.jpeg
- WhatsApp Image 2026-04-22 at 7.29.15 AM.jpeg
- WhatsApp Image 2026-04-22 at 7.31.33 AM.jpeg
- WhatsApp Image 2026-04-22 at 9.59.51 AM.jpeg
- WhatsApp Image 2026-04-23 at 3.07.23 AM (1).jpeg
- WhatsApp Image 2026-04-23 at 9.56.04 AM.jpeg
- WhatsApp Image 2026-04-24 at 10.23.05 AM.jpeg
- WhatsApp Image 2026-04-24 at 8.07.50 AM.jpeg
- WhatsApp Image 2026-04-24 at 8.53.54 AM.jpeg
- WhatsApp Image 2026-04-24 at 9.14.10 AM.jpeg


In [17]:
canonical_classes = [
    class_name.replace(" ", "_")
    for class_name in EXPECTED_COUNTS.keys()
]

classes_file = PROJECT_ROOT / "CLASSES.md"

content = "# CESPPL Canonical Classes\n\n"

for class_name in canonical_classes:
    content += f"- {class_name}\n"

classes_file.write_text(content, encoding="utf-8")

print("CLASSES.md created successfully")
print("Saved at:", classes_file)
print()
print(classes_file.read_text(encoding="utf-8"))

CLASSES.md created successfully
Saved at: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\CLASSES.md

# CESPPL Canonical Classes

- BIN_LIFTING
- BIN_WASHING
- GATE_MEETING
- LFC
- MANUAL_BEACH_CLEANING
- MECHANICAL_SWEEPING
- MECHANIZED_BEACH_CLEANING
- PRIMARY_COLLECTION
- ROAD_SWEEPING
- SECONDARY_VEHICLES



In [18]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".tif",
    ".tiff",
    ".webp",
    ".heic",
    ".heif"
}

SYSTEM_FILES = {
    "thumbs.db",
    "desktop.ini",
    ".ds_store"
}

print("Image extensions and system files configured")

Image extensions and system files configured


In [19]:
def is_hidden_file(file_path, class_folder):
    try:
        relative_parts = file_path.relative_to(class_folder).parts
    except ValueError:
        relative_parts = file_path.parts

    return any(part.startswith(".") for part in relative_parts)


def dimension_area(dimension):
    width, height = dimension
    return width * height


def format_dimension(width, height):
    return f"{round(width)}x{round(height)}"


print("Helper functions created successfully")

Helper functions created successfully


In [20]:
inventory_rows = []
extension_rows = []
excluded_rows = []
file_size_rows = []

for class_name in EXPECTED_COUNTS.keys():

    class_folder = RAW_DATASET / class_name

    all_files = sorted(
        file_path
        for file_path in class_folder.rglob("*")
        if file_path.is_file()
    )

    extension_counter = Counter()

    valid_dimensions = []
    valid_file_sizes_mb = []

    corrupt_count = 0
    non_image_count = 0
    hidden_system_count = 0

    for file_path in all_files:

        extension = file_path.suffix.lower()

        if not extension:
            extension = "[no extension]"

        extension_counter[extension] += 1

        relative_path = file_path.relative_to(RAW_DATASET)

        if (
            file_path.name.lower() in SYSTEM_FILES
            or is_hidden_file(file_path, class_folder)
        ):
            hidden_system_count += 1

            excluded_rows.append({
                "class": class_name.replace(" ", "_"),
                "file": str(relative_path),
                "extension": extension,
                "reason": "hidden_or_system_file",
                "error": ""
            })

            continue

        try:
            with Image.open(file_path) as image:
                image.load()

                width, height = image.size

                if width <= 0 or height <= 0:
                    raise ValueError(
                        f"Invalid dimensions: {width}x{height}"
                    )

                valid_dimensions.append((width, height))

                file_size_mb = file_path.stat().st_size / (1024 * 1024)
                valid_file_sizes_mb.append(file_size_mb)

        except (
            UnidentifiedImageError,
            OSError,
            ValueError,
            SyntaxError
        ) as error:

            if file_path.suffix.lower() in IMAGE_EXTENSIONS:
                corrupt_count += 1
                reason = "corrupt_or_unreadable_image"
            else:
                non_image_count += 1
                reason = "non_image_file"

            excluded_rows.append({
                "class": class_name.replace(" ", "_"),
                "file": str(relative_path),
                "extension": extension,
                "reason": reason,
                "error": str(error)[:250]
            })

    for extension, count in sorted(extension_counter.items()):
        extension_rows.append({
            "class": class_name.replace(" ", "_"),
            "extension": extension,
            "count": count
        })

    if valid_dimensions:

        smallest_dimension = min(
            valid_dimensions,
            key=dimension_area
        )

        largest_dimension = max(
            valid_dimensions,
            key=dimension_area
        )

        mean_width = statistics.mean(
            width for width, height in valid_dimensions
        )

        mean_height = statistics.mean(
            height for width, height in valid_dimensions
        )

        smallest_dim = format_dimension(
            smallest_dimension[0],
            smallest_dimension[1]
        )

        largest_dim = format_dimension(
            largest_dimension[0],
            largest_dimension[1]
        )

        mean_dim = format_dimension(
            mean_width,
            mean_height
        )

    else:
        smallest_dim = ""
        largest_dim = ""
        mean_dim = ""

    inventory_rows.append({
        "class": class_name.replace(" ", "_"),
        "total_files": len(all_files),
        "valid_images": len(valid_dimensions),
        "corrupt": corrupt_count,
        "smallest_dim": smallest_dim,
        "largest_dim": largest_dim,
        "mean_dim": mean_dim
    })

    if valid_file_sizes_mb:
        file_size_rows.append({
            "class": class_name.replace(" ", "_"),
            "smallest_file_mb": round(min(valid_file_sizes_mb), 3),
            "largest_file_mb": round(max(valid_file_sizes_mb), 3),
            "mean_file_mb": round(
                statistics.mean(valid_file_sizes_mb),
                3
            ),
            "median_file_mb": round(
                statistics.median(valid_file_sizes_mb),
                3
            )
        })

    print(
        f"{class_name}: "
        f"{len(valid_dimensions)} valid, "
        f"{corrupt_count} corrupt, "
        f"{non_image_count} non-image, "
        f"{hidden_system_count} hidden/system"
    )

print("\nDataset scan completed successfully")

BIN LIFTING: 166 valid, 0 corrupt, 0 non-image, 0 hidden/system
BIN WASHING: 327 valid, 0 corrupt, 0 non-image, 0 hidden/system
GATE MEETING: 367 valid, 0 corrupt, 0 non-image, 0 hidden/system
LFC: 217 valid, 0 corrupt, 0 non-image, 0 hidden/system
MANUAL BEACH CLEANING: 1384 valid, 0 corrupt, 0 non-image, 0 hidden/system
MECHANICAL SWEEPING: 284 valid, 0 corrupt, 0 non-image, 0 hidden/system
MECHANIZED BEACH CLEANING: 302 valid, 0 corrupt, 0 non-image, 0 hidden/system
PRIMARY COLLECTION: 208 valid, 0 corrupt, 0 non-image, 0 hidden/system
ROAD SWEEPING: 530 valid, 0 corrupt, 0 non-image, 0 hidden/system
SECONDARY VEHICLES: 394 valid, 0 corrupt, 0 non-image, 0 hidden/system

Dataset scan completed successfully


In [21]:
inventory_df = pd.DataFrame(inventory_rows)

extension_df = pd.DataFrame(extension_rows)

excluded_df = pd.DataFrame(
    excluded_rows,
    columns=[
        "class",
        "file",
        "extension",
        "reason",
        "error"
    ]
)

file_size_df = pd.DataFrame(file_size_rows)

print("Inventory table")
display(inventory_df)

Inventory table


,class,total_files,valid_images,corrupt,smallest_dim,largest_dim,mean_dim
0,BIN_LIFTING,166,166,0,667x543,1204x1600,886x1085
1,BIN_WASHING,327,327,0,637x477,1600x1200,1284x1164
2,GATE_MEETING,367,367,0,667x542,1600x1600,1125x869
3,LFC,217,217,0,666x543,1200x1600,1085x1204
4,MANUAL_BEACH_CLEANING,1384,1384,0,667x542,1600x1200,1216x1332
5,MECHANICAL_SWEEPING,284,284,0,666x543,1204x1600,912x1087
6,MECHANIZED_BEACH_CLEANING,302,302,0,667x542,1200x1600,896x877
7,PRIMARY_COLLECTION,208,208,0,667x542,1200x1600,631x827
8,ROAD_SWEEPING,530,530,0,667x542,1599x1599,1176x1197
9,SECONDARY_VEHICLES,394,394,0,666x543,1600x1200,1052x909


In [22]:
if extension_df.empty:
    print("No file extensions were found")
else:
    extension_pivot = (
        extension_df
        .pivot_table(
            index="class",
            columns="extension",
            values="count",
            aggfunc="sum",
            fill_value=0
        )
        .astype(int)
    )

    display(extension_pivot)

extension,.jpeg,.png
class,,
BIN_LIFTING,75,91
BIN_WASHING,267,60
GATE_MEETING,367,0
LFC,217,0
MANUAL_BEACH_CLEANING,1383,1
MECHANICAL_SWEEPING,284,0
MECHANIZED_BEACH_CLEANING,300,2
PRIMARY_COLLECTION,30,178
ROAD_SWEEPING,530,0


In [23]:
print("File size distribution in MB")
display(file_size_df)

File size distribution in MB


,class,smallest_file_mb,largest_file_mb,mean_file_mb,median_file_mb
0,BIN_LIFTING,0.101,1.165,0.542,0.651
1,BIN_WASHING,0.103,1.053,0.327,0.233
2,GATE_MEETING,0.057,0.725,0.186,0.173
3,LFC,0.081,0.352,0.221,0.214
4,MANUAL_BEACH_CLEANING,0.039,1.114,0.203,0.192
5,MECHANICAL_SWEEPING,0.069,0.314,0.144,0.139
6,MECHANIZED_BEACH_CLEANING,0.054,0.466,0.153,0.142
7,PRIMARY_COLLECTION,0.144,1.068,0.713,0.768
8,ROAD_SWEEPING,0.093,0.382,0.213,0.202
9,SECONDARY_VEHICLES,0.071,0.294,0.159,0.157


In [24]:
if excluded_df.empty:
    print("No corrupt, non-image, hidden or system files were found")
else:
    print("Total excluded files:", len(excluded_df))
    display(excluded_df)

No corrupt, non-image, hidden or system files were found


In [25]:
comparison_rows = []

for class_name, expected_count in EXPECTED_COUNTS.items():

    canonical_name = class_name.replace(" ", "_")

    matching_row = inventory_df[
        inventory_df["class"] == canonical_name
    ]

    actual_count = int(
        matching_row.iloc[0]["valid_images"]
    )

    difference = actual_count - expected_count

    comparison_rows.append({
        "class": canonical_name,
        "expected_images": expected_count,
        "actual_valid_images": actual_count,
        "difference": difference,
        "status": "MATCH" if difference == 0 else "CHECK"
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,class,expected_images,actual_valid_images,difference,status
0,BIN_LIFTING,131,166,35,CHECK
1,BIN_WASHING,327,327,0,MATCH
2,GATE_MEETING,367,367,0,MATCH
3,LFC,217,217,0,MATCH
4,MANUAL_BEACH_CLEANING,1384,1384,0,MATCH
5,MECHANICAL_SWEEPING,284,284,0,MATCH
6,MECHANIZED_BEACH_CLEANING,302,302,0,MATCH
7,PRIMARY_COLLECTION,208,208,0,MATCH
8,ROAD_SWEEPING,530,530,0,MATCH
9,SECONDARY_VEHICLES,394,394,0,MATCH


In [26]:
expected_total = sum(EXPECTED_COUNTS.values())
actual_total = int(inventory_df["valid_images"].sum())

print("Expected valid images :", expected_total)
print("Actual valid images   :", actual_total)
print("Difference            :", actual_total - expected_total)

print()

if actual_total == expected_total:
    print("All counts match the documentation")
else:
    print("The received dataset differs from the documented counts")
    print("The discrepancy has already been investigated")

Expected valid images : 4144
Actual valid images   : 4179
Difference            : 35

The received dataset differs from the documented counts
The discrepancy has already been investigated


In [27]:
inventory_path = PROJECT_ROOT / "inventory.csv"

required_columns = [
    "class",
    "total_files",
    "valid_images",
    "corrupt",
    "smallest_dim",
    "largest_dim",
    "mean_dim"
]

inventory_df[required_columns].to_csv(
    inventory_path,
    index=False,
    encoding="utf-8"
)

print("inventory.csv saved successfully")
print("Saved at:", inventory_path)

inventory.csv saved successfully
Saved at: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\inventory.csv


In [28]:
excluded_path = PROJECT_ROOT / "excluded_files.csv"
extension_path = PROJECT_ROOT / "extension_counts.csv"
file_size_path = PROJECT_ROOT / "file_size_summary.csv"

excluded_df.to_csv(
    excluded_path,
    index=False,
    encoding="utf-8"
)

extension_df.to_csv(
    extension_path,
    index=False,
    encoding="utf-8"
)

file_size_df.to_csv(
    file_size_path,
    index=False,
    encoding="utf-8"
)

print("Supporting reports saved successfully")
print(excluded_path)
print(extension_path)
print(file_size_path)

Supporting reports saved successfully
C:\Users\sarun\OneDrive\Desktop\cesppl-internship\excluded_files.csv
C:\Users\sarun\OneDrive\Desktop\cesppl-internship\extension_counts.csv
C:\Users\sarun\OneDrive\Desktop\cesppl-internship\file_size_summary.csv


In [29]:
saved_inventory = pd.read_csv(
    PROJECT_ROOT / "inventory.csv"
)

print("Rows:", len(saved_inventory))
print("Columns:", list(saved_inventory.columns))

display(saved_inventory)

Rows: 10
Columns: ['class', 'total_files', 'valid_images', 'corrupt', 'smallest_dim', 'largest_dim', 'mean_dim']


,class,total_files,valid_images,corrupt,smallest_dim,largest_dim,mean_dim
0,BIN_LIFTING,166,166,0,667x543,1204x1600,886x1085
1,BIN_WASHING,327,327,0,637x477,1600x1200,1284x1164
2,GATE_MEETING,367,367,0,667x542,1600x1600,1125x869
3,LFC,217,217,0,666x543,1200x1600,1085x1204
4,MANUAL_BEACH_CLEANING,1384,1384,0,667x542,1600x1200,1216x1332
5,MECHANICAL_SWEEPING,284,284,0,666x543,1204x1600,912x1087
6,MECHANIZED_BEACH_CLEANING,302,302,0,667x542,1200x1600,896x877
7,PRIMARY_COLLECTION,208,208,0,667x542,1200x1600,631x827
8,ROAD_SWEEPING,530,530,0,667x542,1599x1599,1176x1197
9,SECONDARY_VEHICLES,394,394,0,666x543,1600x1200,1052x909


# Day Summary

The CESPPL field dataset was preserved inside `data/cesppl_raw` without modifying the received raw files. The complete folder structure, file formats, file sizes, image dimensions, corrupt files, hidden files and non-image files were examined.

The documentation expected ten classes containing 4,144 images. However, the received dataset contains 4,179 valid images.

The discrepancy is in the `BIN_LIFTING` class. The documentation expected 131 images, while the received folder contains 166 valid images. All 166 files were successfully opened as images. No corrupt files, non-image files or exact duplicate files were found. Therefore, the inventory records the actual dataset received from the mentor rather than manually forcing the documented count.

Based on the documented counts, the largest class is `MANUAL_BEACH_CLEANING` with 1,384 images, which is approximately 10.6 times the documented smallest class, `BIN_LIFTING`, with 131 images.

Based on the received dataset, `MANUAL_BEACH_CLEANING` is approximately 8.3 times the size of `BIN_LIFTING`, because the received `BIN_LIFTING` folder contains 166 images.

The strong class imbalance means that accuracy alone may provide a misleading measure of model performance. Class weighting should be used from the beginning, while macro-F1 and per-class recall should be treated as the main evaluation metrics.